## Understand the three separate actions

---

| Action   | What happens                                         | Do you need to repeat it?                      |
| -------- | ---------------------------------------------------- | ---------------------------------------------- |
| Download | Yahoo sends data into a Python variable such as `df` | Only when you want new data                    |
| Save     | Python writes the data into a file on your computer  | After downloading data you want to keep        |
| Load     | Python reads that saved file into `df`               | Whenever you restart Python and want to use it |


A DataFrame lives in your running Python session. **Saving the notebook does not reliably preserve that variable for your next session**. A CSV saves the actual rows of data separately.

Once saved, the CSV remains usable even after Yahoo stops offering that period.

## How much Yahoo data can you download?

`interval` means the duration represented by each row. `period` means how far back you request data.

For example:

```python
interval="5m"   # Each row represents five minutes
period="1mo"    # Request approximately one month
```

| Interval                                  | Practical availability                                                                                   |
| ----------------------------------------- | -------------------------------------------------------------------------------------------------------- |
| `"1m"`                                    | Short recent windows. Start with five days; current yfinance code uses eight days for a maximum request. |
| `"2m"`, `"5m"`, `"15m"`, `"30m"`, `"90m"` | Generally within the most recent 60 days                                                                 |
| `"1h"` or `"60m"`                         | Generally up to approximately 730 days                                                                   |
| `"1d"`, `"1wk"`, `"1mo"`                  | Often many years, depending on the instrument                                                            |


## Organise the notebooks and saved data

Inside your existing LEAN project, use:

| Item                  | Purpose                                            |
| --------------------- | -------------------------------------------------- |
| `download_data.ipynb` | Download, inspect, and save Yahoo data             |
| `research.ipynb`      | Load data, calculate indicators, and create charts |
| `research-data/`      | Store your downloaded CSV files                    |
| `main.py`             | Your LEAN backtesting algorithm                    |


Keep both notebooks at the project’s top level for these examples.

You can run the download notebook using your normal Windows .venv, like the one shown in your screenshots. The research notebook can read the resulting file from either ordinary Jupyter or LEAN research.

---

## Downloading and Cleaning Yahoo Finance Data with `yfinance`

In [34]:
from pathlib import Path
import pandas as pd 
import yfinance as yf

In [35]:
TICKER = "QQQ"

df = yf.download(
    TICKER,
    period="5d", 
    interval="1m",
    auto_adjust=True,
    prepost=False, 
    multi_level_index=False,
    progress=False,
)

In [36]:
df.head()

,Close,High,Low,Open,Volume
Datetime,,,,,
2026-08-31 09:30:00-04:00,716.110107,716.110107,714.679993,715.119995,1290381
2026-08-31 09:31:00-04:00,716.799988,717.000977,716.109985,716.135010,123098
2026-08-31 09:32:00-04:00,715.849976,716.930115,715.809998,716.804993,134898
2026-08-31 09:33:00-04:00,715.590027,716.070007,715.020020,715.849976,166824
2026-08-31 09:34:00-04:00,715.799927,715.809998,715.369995,715.530029,127281


In [37]:
# access the columns, treat those columns as strings and convert them to lowercase
df.columns = df.columns.str.lower()
df.head(1)

,close,high,low,open,volume
Datetime,,,,,
2026-08-31 09:30:00-04:00,716.110107,716.110107,714.679993,715.119995,1290381


In [38]:
# reorder the data in df 
df = df[["open", "high", "low", "close", "volume"]]
df.head(1)

,open,high,low,close,volume
Datetime,,,,,
2026-08-31 09:30:00-04:00,715.119995,716.110107,714.679993,716.110107,1290381


In [39]:
# convert the index to UTC timezone
df.index = df.index.tz_convert("UTC")
df.head(2)

,open,high,low,close,volume
Datetime,,,,,
2026-08-31 13:30:00+00:00,715.119995,716.110107,714.679993,716.110107,1290381
2026-08-31 13:31:00+00:00,716.135010,717.000977,716.109985,716.799988,123098


In [40]:
# Quant Connect works with the bars end time.
bar_length = pd.Timedelta(minutes=1)
df.index = df.index + bar_length
df.head(2)


,open,high,low,close,volume
Datetime,,,,,
2026-08-31 13:31:00+00:00,715.119995,716.110107,714.679993,716.110107,1290381
2026-08-31 13:32:00+00:00,716.135010,717.000977,716.109985,716.799988,123098


In [41]:
# match the naming to Quant Connect
df.index.name = "time"
df.head(1)

,open,high,low,close,volume
time,,,,,
2026-08-31 13:31:00+00:00,715.119995,716.110107,714.679993,716.110107,1290381


In [ ]:
# Organise the data by sorting the index
df = df.sort_index()
df.head()

,open,high,low,close,volume
time,,,,,
2026-08-31 13:31:00+00:00,715.119995,716.110107,714.679993,716.110107,1290381
2026-08-31 13:32:00+00:00,716.135010,717.000977,716.109985,716.799988,123098
2026-08-31 13:33:00+00:00,716.804993,716.930115,715.809998,715.849976,134898
2026-08-31 13:34:00+00:00,715.849976,716.070007,715.020020,715.590027,166824
2026-08-31 13:35:00+00:00,715.530029,715.809998,715.369995,715.799927,127281


In [49]:
print(f"Downloaded {len(df):,} rows")
print("First timestamp:", df.index.min())
print("Last timestamp:" , df.index.max())

Downloaded 1,949 rows
First timestamp: 2026-08-31 13:31:00+00:00
Last timestamp: 2026-09-04 20:00:00+00:00


# Process 

```python
# Make the column names lowercase.
df.columns = df.columns.str.lower()

# Select the OHLCV columns and put them in a consistent order.
df = df[["open", "high", "low", "close", "volume"]]

# Convert the timestamps to UTC.
df.index = df.index.tz_convert("UTC")

# A one-minute candle lasts one minute.
bar_length = pd.Timedelta(minutes=1)

# Move Yahoo's start-time label to the candle's end time.
df.index = df.index + bar_length

# Rename the datetime index.
df.index.name = "time"

# Sort from oldest to newest.
df = df.sort_index()

df.head()
```